# Module 16 — Iterators, Generators, and Context Managers

**Part IV: Advanced** · Level: Advanced · Prerequisites: Modules 5, 11, 14

## What you'll learn

- The iterable / iterator protocol
- Writing generators with `yield`
- Lazy pipelines and `yield from`
- Context managers: `with`, `__enter__` / `__exit__`, and `@contextmanager`

---

## 16.1 Iterables and iterators

Two related ideas power every `for` loop:

- An **iterable** can produce an iterator: it has `__iter__()` (lists, strings, dicts, files, ranges).
- An **iterator** produces values one at a time: it has `__next__()` and raises `StopIteration` when exhausted.

In [ ]:
numbers = [1, 2, 3]
print(hasattr(numbers, "__iter__"), hasattr(numbers, "__next__"))   # True False

it = iter(numbers)            # get an iterator from the iterable
print(next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("exhausted")

print(next(iter([]), "default"))      # next() with a fallback value

A `for` loop is exactly: call `iter()`, call `next()` repeatedly, stop on `StopIteration`.

## 16.2 Writing an iterator class

In [ ]:
class Countdown:
    """Iterate from `start` down to 1."""

    def __init__(self, start):
        self.start = start

    def __iter__(self):
        self.current = self.start
        return self                     # the object is its own iterator

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

print(list(Countdown(3)))               # [3, 2, 1]

## 16.3 Generators

A **generator function** contains `yield`. Calling it does not run the body; it returns a generator (an iterator) that runs *until the next `yield`*, pauses with all local state intact, and resumes on the next request. It writes the iterator class above in a few lines.

In [ ]:
def squares(n):
    for i in range(n):
        yield i * i

print(list(squares(5)))                 # [0, 1, 4, 9, 16]

def countdown(n):
    while n > 0:
        yield n
        n -= 1

print(list(countdown(3)))

gen = countdown(2)
print(next(gen), next(gen))

Watch the pausing in action:

In [ ]:
def chatty():
    print("start")
    yield 1
    print("middle")
    yield 2
    print("end")

g = chatty()
print("created")
print(next(g))
print(next(g))

## 16.4 Why generators matter: laziness

Values are produced **on demand**, so memory stays tiny and infinite sequences are possible.

In [ ]:
def naturals():
    n = 0
    while True:
        yield n
        n += 1

from itertools import islice
print(list(islice(naturals(), 5)))      # [0, 1, 2, 3, 4]

def read_big_file(lines):
    for line in lines:                  # imagine this is a 10 GB file
        yield line.strip()

### Pipelines

Chain generators to process data in streaming fashion — each stage handles one item at a time.

In [ ]:
def numbers():
    yield from range(1, 11)

def evens(source):
    return (x for x in source if x % 2 == 0)

def squared(source):
    return (x * x for x in source)

print(sum(squared(evens(numbers()))))   # 4 + 16 + 36 + 64 + 100 = 220

### `yield from`

`yield from` delegates to another iterable.

In [ ]:
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)
        else:
            yield item

print(list(flatten([1, [2, [3, 4]], 5])))

### Generators are single-use

In [ ]:
g = squares(3)
print(list(g), list(g))                 # [0, 1, 4] []

## 16.5 Context managers: the `with` statement

`with` guarantees **setup and cleanup** happen, even if an exception occurs. You already used it with files:

In [ ]:
from pathlib import Path
import tempfile

path = Path(tempfile.mkdtemp()) / "demo.txt"
with open(path, "w") as f:
    f.write("hi")
print(f.closed)                         # True -- closed automatically

### Writing one with a class

An object is a context manager if it defines `__enter__` and `__exit__`.

In [ ]:
import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self                      # value bound by `as`

    def __exit__(self, exc_type, exc, traceback):
        self.elapsed = time.perf_counter() - self.start
        print(f"block took {self.elapsed:.4f}s")
        return False                     # False = do NOT swallow exceptions

with Timer() as t:
    total = sum(range(100_000))
print(total, t.elapsed >= 0)

try:
    with Timer():
        raise ValueError("boom")         # __exit__ still runs
except ValueError:
    print("error propagated")

If `__exit__` returns `True`, it suppresses the exception — do that only deliberately.

### The shortcut: `@contextmanager`

In [ ]:
from contextlib import contextmanager

@contextmanager
def open_session(name):
    print(f"open {name}")
    try:
        yield name                       # the body of the `with` runs here
    finally:
        print(f"close {name}")           # cleanup always runs

with open_session("db") as s:
    print("using", s)

### Useful ready-made managers

In [ ]:
from contextlib import suppress, redirect_stdout
import io, os

with suppress(FileNotFoundError):        # ignore one specific error
    os.remove("this-file-does-not-exist.txt")

buf = io.StringIO()
with redirect_stdout(buf):               # capture print output
    print("captured")
print(repr(buf.getvalue()))

from pathlib import Path
with open(Path(tempfile.mkdtemp()) / "a.txt", "w") as a, Timer():   # several at once
    a.write("x")

## Common pitfalls

- Expecting a generator to be reusable.
- Forgetting `try`/`finally` around `yield` in a `@contextmanager`, so cleanup is skipped on error.
- Returning `True` from `__exit__` and accidentally hiding bugs.
- Building a full list when a generator expression would do.
- Mutating a collection while iterating over its iterator.

## Practice

1. Write a generator `fibonacci()` that yields forever, then use `islice` to print the first ten.
2. Write a generator `read_chunks(sequence, size)` that yields successive slices.
3. Write a context manager `working_directory(path)` that changes directory and restores the original on exit.